# Create train, validation, and test splits

This notebook reads `Data/Raw/Descriptor.csv` and writes split metadata into `Data/Preprocessed`.

It does not copy or move matrix `.txt` files. Later notebooks can use the saved `date` and `cell_id` values to load files from `Data/Raw/Annotated_data/<date>/`.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    (PROJECT_ROOT / "Source").exists()
    or (PROJECT_ROOT / "Data").exists()
    or (PROJECT_ROOT / "pmaciag-volume-beach").exists()
):
    PROJECT_ROOT = PROJECT_ROOT.parent
if (PROJECT_ROOT / "pmaciag-volume-beach").exists():
    PROJECT_ROOT = PROJECT_ROOT / "pmaciag-volume-beach"
if not ((PROJECT_ROOT / "Source").exists() or (PROJECT_ROOT / "Data").exists()):
    raise RuntimeError("Could not locate pmaciag-volume-beach project directory.")

DESCRIPTOR_PATH = PROJECT_ROOT / "Data" / "Raw" / "Descriptor.csv"
PREPROCESSED_DIR = PROJECT_ROOT / "Data" / "Preprocessed"

# Fractions are applied to descriptor rows. Keep their sum equal to 1.0.
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15
RANDOM_SEED = 42

assert abs((TRAIN_FRACTION + VALIDATION_FRACTION + TEST_FRACTION) - 1.0) < 1e-9

print(f"Project root:     {PROJECT_ROOT}")
print(f"Descriptor:       {DESCRIPTOR_PATH}")
print(f"Output directory: {PREPROCESSED_DIR}")

In [ ]:
if not DESCRIPTOR_PATH.exists():
    raise FileNotFoundError(f"Descriptor file not found: {DESCRIPTOR_PATH}")

descriptor = pd.read_csv(
    DESCRIPTOR_PATH,
    sep=";",
    header=None,
    names=["date", "cell_id", "class"],
    dtype={"date": "string", "cell_id": "string", "class": "string"},
    encoding="utf-8-sig",
)

descriptor = descriptor.dropna(subset=["date", "cell_id", "class"]).copy()
descriptor["date"] = descriptor["date"].str.strip()
descriptor["cell_id"] = descriptor["cell_id"].str.strip()
descriptor["class"] = descriptor["class"].str.strip()

duplicate_mask = descriptor.duplicated(subset=["date", "cell_id"], keep=False)
if duplicate_mask.any():
    duplicates = descriptor.loc[duplicate_mask].sort_values(["date", "cell_id"])
    raise ValueError(
        "Descriptor contains duplicate date/cell_id pairs. "
        "Resolve duplicates before creating splits."
    )

descriptor = descriptor.sort_values(["date", "cell_id"]).reset_index(drop=True)
descriptor.head(), descriptor.shape

In [ ]:
class_counts = descriptor["class"].value_counts().sort_index()
class_counts.to_frame(name="count")

In [ ]:
train_df, holdout_df = train_test_split(
    descriptor,
    train_size=TRAIN_FRACTION,
    random_state=RANDOM_SEED,
    shuffle=True,
    stratify=descriptor["class"],
)

validation_relative_fraction = VALIDATION_FRACTION / (VALIDATION_FRACTION + TEST_FRACTION)
validation_df, test_df = train_test_split(
    holdout_df,
    train_size=validation_relative_fraction,
    random_state=RANDOM_SEED,
    shuffle=True,
    stratify=holdout_df["class"],
)

train_df = train_df.assign(split="train")
validation_df = validation_df.assign(split="validation")
test_df = test_df.assign(split="test")

split_df = pd.concat([train_df, validation_df, test_df], ignore_index=True)
split_df = split_df.sort_values(["split", "date", "cell_id"]).reset_index(drop=True)

split_counts = split_df["split"].value_counts().reindex(["train", "validation", "test"])
split_counts.to_frame(name="count")

In [ ]:
expected_keys = set(map(tuple, descriptor[["date", "cell_id"]].to_numpy()))
actual_keys = set(map(tuple, split_df[["date", "cell_id"]].to_numpy()))

if expected_keys != actual_keys:
    missing = expected_keys - actual_keys
    unexpected = actual_keys - expected_keys
    raise AssertionError(f"Split mismatch. Missing={len(missing)}, unexpected={len(unexpected)}")

if split_df.duplicated(subset=["date", "cell_id"]).any():
    raise AssertionError("A date/cell_id pair appears in more than one split.")

distribution = (
    split_df
    .groupby(["split", "class"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(["train", "validation", "test"])
)

distribution

In [ ]:
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

columns = ["date", "cell_id", "class", "split"]
output_files = {
    "all": PREPROCESSED_DIR / "dataset_splits.csv",
    "train": PREPROCESSED_DIR / "train_split.csv",
    "validation": PREPROCESSED_DIR / "validation_split.csv",
    "test": PREPROCESSED_DIR / "test_split.csv",
}

split_df[columns].to_csv(output_files["all"], index=False)
for split_name in ["train", "validation", "test"]:
    split_df.loc[split_df["split"] == split_name, columns].to_csv(output_files[split_name], index=False)

for split_name, path in output_files.items():
    print(f"{split_name:>10}: {path}")